# 02 — Train, Evaluate & Compare YOLO cho Traffic Sign Detection

Notebook này được thiết kế để **treo máy chạy nhiều thí nghiệm liên tiếp** trên RTX 4060, thay vì chỉ train một model rồi dừng.

Mục tiêu chính:

- Kiểm tra môi trường GPU trước khi train.
- Tạo một `data_local.yaml` có đường dẫn chắc chắn đúng trên máy local.
- Chạy 4 cấu hình tạo thành ma trận **model size × input resolution**:
  - YOLO11n — 640
  - YOLO11n — 320
  - YOLO11s — 640
  - YOLO11s — 320
- Mỗi cấu hình được train với tối đa 100 epoch và Early Stopping.
- Tự động evaluate trên cả **validation** và **test**.
- Lưu `best.pt`, metric tổng hợp và metric theo từng class.
- Nếu một experiment lỗi, notebook ghi lỗi và tiếp tục experiment kế tiếp.
- Có cơ chế resume cơ bản từ `last.pt` nếu phiên train trước bị gián đoạn.

> Trước khi chạy qua đêm: nên đóng Android Studio, LDPlayer, game/Overwolf và các ứng dụng dùng GPU không cần thiết. Đồng thời đặt Windows **không Sleep khi cắm điện**.


## 1. Kiểm tra môi trường

Cell này xác nhận notebook đang dùng đúng `.venv`, PyTorch nhận CUDA và GPU là RTX 4060. Nếu `CUDA available = False` thì không nên chạy các cell train bên dưới.


In [1]:
# Kiểm tra Python, Ultralytics và GPU
import sys
import torch
import ultralytics

print("Python:", sys.version.split()[0])
print("Python executable:", sys.executable)
print("PyTorch:", torch.__version__)
print("Ultralytics:", ultralytics.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("CUDA runtime:", torch.version.cuda)
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")


Python: 3.13.2
Python executable: d:\Project\TrafficSignAI\.venv\Scripts\python.exe
PyTorch: 2.14.0+cu126
Ultralytics: 8.4.142
CUDA available: True
CUDA runtime: 12.6
GPU: NVIDIA GeForce RTX 4060
VRAM: 8.0 GB


## 2. Xác định project root và dataset

Notebook có thể được mở từ thư mục `notebooks/` hoặc từ project root. Cell này tự tìm `TrafficSignAI` dựa trên sự tồn tại của thư mục `VR-TSD-2`.


In [2]:
# Tìm project root một cách tương đối an toàn
from pathlib import Path

cwd = Path.cwd().resolve()

candidates = [cwd, cwd.parent, cwd.parent.parent]
PROJECT_ROOT = None

for candidate in candidates:
    if (candidate / "VR-TSD-2").exists():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError("Không tìm thấy thư mục VR-TSD-2 từ working directory hiện tại.")

DATASET_ROOT = PROJECT_ROOT / "VR-TSD-2"
SOURCE_YAML = DATASET_ROOT / "data.yaml"
RUNS_ROOT = PROJECT_ROOT / "runs" / "overnight_compare"
MODELS_ROOT = PROJECT_ROOT / "models"

RUNS_ROOT.mkdir(parents=True, exist_ok=True)
MODELS_ROOT.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT :", PROJECT_ROOT)
print("DATASET_ROOT :", DATASET_ROOT)
print("SOURCE_YAML  :", SOURCE_YAML)
print("RUNS_ROOT    :", RUNS_ROOT)
print("MODELS_ROOT  :", MODELS_ROOT)


PROJECT_ROOT : D:\Project\TrafficSignAI
DATASET_ROOT : D:\Project\TrafficSignAI\VR-TSD-2
SOURCE_YAML  : D:\Project\TrafficSignAI\VR-TSD-2\data.yaml
RUNS_ROOT    : D:\Project\TrafficSignAI\runs\overnight_compare
MODELS_ROOT  : D:\Project\TrafficSignAI\models


## 3. Tạo `data_local.yaml`

File Roboflow hiện tại dùng đường dẫn dạng `../train/images`. Để tránh phụ thuộc cách Ultralytics resolve relative path trên từng môi trường, cell này tạo một YAML local mới với `path` tuyệt đối và các split tương đối rõ ràng.

File gốc `data.yaml` được giữ nguyên.


In [3]:
# Tạo YAML local dùng riêng cho máy này
import yaml

with open(SOURCE_YAML, "r", encoding="utf-8") as f:
    source_data = yaml.safe_load(f)

local_data = {
    "path": str(DATASET_ROOT).replace("\\", "/"),
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "nc": source_data["nc"],
    "names": source_data["names"],
}

LOCAL_YAML = DATASET_ROOT / "data_local.yaml"

with open(LOCAL_YAML, "w", encoding="utf-8") as f:
    yaml.safe_dump(local_data, f, allow_unicode=True, sort_keys=False)

print("Đã tạo:", LOCAL_YAML)
print("Số class:", local_data["nc"])
print("Train path:", DATASET_ROOT / local_data["train"])
print("Val path  :", DATASET_ROOT / local_data["val"])
print("Test path :", DATASET_ROOT / local_data["test"])


Đã tạo: D:\Project\TrafficSignAI\VR-TSD-2\data_local.yaml
Số class: 58
Train path: D:\Project\TrafficSignAI\VR-TSD-2\train\images
Val path  : D:\Project\TrafficSignAI\VR-TSD-2\valid\images
Test path : D:\Project\TrafficSignAI\VR-TSD-2\test\images


## 4. Kiểm tra nhanh dataset trước khi train

Đây là kiểm tra tối thiểu để tránh treo máy nhiều giờ rồi mới phát hiện đường dẫn sai. Cell chỉ xác nhận các thư mục ảnh/label tồn tại và đếm số file.


In [4]:
# Kiểm tra số lượng ảnh và label của từng split
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

for split in ["train", "valid", "test"]:
    image_dir = DATASET_ROOT / split / "images"
    label_dir = DATASET_ROOT / split / "labels"

    images = [p for p in image_dir.iterdir() if p.suffix.lower() in IMAGE_EXTENSIONS]
    labels = list(label_dir.glob("*.txt"))

    print(f"{split.upper():5s} | images = {len(images):4d} | labels = {len(labels):4d}")

    if not image_dir.exists() or not label_dir.exists():
        raise FileNotFoundError(f"Thiếu thư mục của split: {split}")


TRAIN | images = 5668 | labels = 5668
VALID | images = 1259 | labels = 1259
TEST  | images = 1151 | labels = 1151


## 5. Danh sách experiment chạy qua đêm

Ta dùng một thiết kế 2×2 để phục vụ đúng mục tiêu deploy mobile:

| Experiment | Model | Input | Ý nghĩa |
|---|---|---:|---|
| `yolo11n_640` | YOLO11n | 640 | Baseline accuracy tốt hơn cho vật thể nhỏ |
| `yolo11n_320` | YOLO11n | 320 | Baseline mobile nhẹ hơn |
| `yolo11s_640` | YOLO11s | 640 | Model lớn hơn để xem accuracy tăng bao nhiêu |
| `yolo11s_320` | YOLO11s | 320 | So sánh model lớn nhưng input nhỏ |

Các experiment dùng cùng seed, cùng số epoch tối đa và cùng Early Stopping để so sánh công bằng hơn.

`batch=-1` để Ultralytics tự chọn batch theo VRAM. `workers=0` ưu tiên độ ổn định trên Windows/Jupyter khi chạy dài.


In [5]:
# Khai báo toàn bộ experiment
EXPERIMENTS = [
    {
        "name": "yolo11n_640",
        "weights": "yolo11n.pt",
        "imgsz": 640,
    },
    {
        "name": "yolo11n_320",
        "weights": "yolo11n.pt",
        "imgsz": 320,
    },
    {
        "name": "yolo11s_640",
        "weights": "yolo11s.pt",
        "imgsz": 640,
    },
    {
        "name": "yolo11s_320",
        "weights": "yolo11s.pt",
        "imgsz": 320,
    },
]

TRAIN_ARGS = {
    "data": str(LOCAL_YAML),
    "epochs": 100,
    "patience": 25,
    "batch": -1,
    "device": 0,
    "workers": 4,
    "cache": False,
    "amp": True,
    "seed": 42,
    "deterministic": False,
    "plots": True,
    "save": True,
    "save_period": 10,
    "verbose": True,
}

for exp in EXPERIMENTS:
    print(exp)


{'name': 'yolo11n_640', 'weights': 'yolo11n.pt', 'imgsz': 640}
{'name': 'yolo11n_320', 'weights': 'yolo11n.pt', 'imgsz': 320}
{'name': 'yolo11s_640', 'weights': 'yolo11s.pt', 'imgsz': 640}
{'name': 'yolo11s_320', 'weights': 'yolo11s.pt', 'imgsz': 320}


## 6. Smoke test ngắn — nên chạy trước khi bỏ máy

Cell này chạy YOLO11n trên một phần nhỏ dữ liệu trong 1 epoch để xác nhận pipeline train thực sự hoạt động trên GPU. Nó không dùng để đánh giá model.

Nếu smoke test chạy xong không lỗi, có thể tiếp tục cell `Overnight training` bên dưới.


In [6]:
# Smoke test 1 epoch để bắt lỗi trước khi train dài
from ultralytics import YOLO

SMOKE_TEST = False

if SMOKE_TEST:
    smoke_model = YOLO("yolo11n.pt")

    smoke_model.train(
        data=str(LOCAL_YAML),
        epochs=1,
        fraction=0.05,
        imgsz=320,
        batch=-1,
        device=0,
        workers=0,
        amp=True,
        project=str(RUNS_ROOT),
        name="_smoke_test",
        exist_ok=True,
        plots=False,
        verbose=True,
    )

    print("Smoke test hoàn tất.")
else:
    print("SMOKE_TEST = False, bỏ qua.")


SMOKE_TEST = False, bỏ qua.


## 7. Hàm hỗ trợ lưu metric

Các hàm dưới đây chuẩn hóa kết quả từ Ultralytics thành CSV để ngày mai có thể đọc nhanh mà không cần mở từng folder `runs`.

Ta lưu:

- Precision
- Recall
- mAP@0.5
- mAP@0.5:0.95
- mAP từng class
- kích thước file `best.pt`
- số parameter
- thời gian train


In [7]:
# Hàm lấy metric an toàn từ Ultralytics
import csv
import json
import shutil
import time
import gc
import numpy as np
import pandas as pd

SUMMARY_CSV = PROJECT_ROOT / "overnight_summary.csv"
ERROR_LOG = PROJECT_ROOT / "overnight_errors.json"

def metric_or_nan(obj, attr):
    value = getattr(obj, attr, None)
    if value is None:
        return float("nan")
    try:
        return float(value)
    except Exception:
        return float("nan")

def extract_box_metrics(metrics):
    box = metrics.box
    return {
        "precision": metric_or_nan(box, "mp"),
        "recall": metric_or_nan(box, "mr"),
        "mAP50": metric_or_nan(box, "map50"),
        "mAP50_95": metric_or_nan(box, "map"),
    }

def save_per_class_metrics(metrics, split_name, experiment_name, class_names):
    box = metrics.box

    maps = np.asarray(getattr(box, "maps", []), dtype=float)
    p = np.asarray(getattr(box, "p", []), dtype=float)
    r = np.asarray(getattr(box, "r", []), dtype=float)
    ap50 = np.asarray(getattr(box, "ap50", []), dtype=float)
    ap = np.asarray(getattr(box, "ap", []), dtype=float)

    rows = []

    for class_id, class_name in enumerate(class_names):
        rows.append({
            "class_id": class_id,
            "class_name": class_name,
            "precision": p[class_id] if class_id < len(p) else np.nan,
            "recall": r[class_id] if class_id < len(r) else np.nan,
            "mAP50": ap50[class_id] if class_id < len(ap50) else np.nan,
            "mAP50_95": maps[class_id] if class_id < len(maps) else (
                ap[class_id] if class_id < len(ap) else np.nan
            ),
        })

    out_csv = RUNS_ROOT / experiment_name / f"per_class_{split_name}.csv"
    pd.DataFrame(rows).to_csv(out_csv, index=False, encoding="utf-8-sig")

    return out_csv

def upsert_summary(row):
    new_df = pd.DataFrame([row])

    if SUMMARY_CSV.exists():
        old_df = pd.read_csv(SUMMARY_CSV)
        old_df = old_df[old_df["experiment"] != row["experiment"]]
        final_df = pd.concat([old_df, new_df], ignore_index=True)
    else:
        final_df = new_df

    final_df.to_csv(SUMMARY_CSV, index=False, encoding="utf-8-sig")
    return final_df


## 8. Overnight training — cell chính

Đây là cell để **treo máy chạy tuần tự**.

Cơ chế:

1. Nếu experiment chưa từng chạy → train mới.
2. Nếu có `last.pt` nhưng chưa có cờ hoàn tất → resume.
3. Nếu train đã hoàn tất → bỏ qua train, chuyển thẳng evaluate.
4. Evaluate `best.pt` trên `val` và `test`.
5. Copy `best.pt` sang folder `models/`.
6. Ghi summary sau từng experiment, nên nếu cell dừng giữa chừng thì kết quả experiment trước vẫn còn.
7. Nếu một experiment lỗi, ghi lỗi vào `overnight_errors.json` rồi tiếp tục experiment kế tiếp.

**Không tắt VS Code/Kernel trong lúc cell này đang chạy.**


In [8]:
# Train + evaluate toàn bộ experiment theo thứ tự
from ultralytics import YOLO

CLASS_NAMES = local_data["names"]
errors = {}

for idx, exp in enumerate(EXPERIMENTS, start=1):
    exp_name = exp["name"]
    weights = exp["weights"]
    imgsz = exp["imgsz"]

    print("\n" + "=" * 90)
    print(f"[{idx}/{len(EXPERIMENTS)}] EXPERIMENT: {exp_name}")
    print("=" * 90)

    exp_dir = RUNS_ROOT / exp_name
    weights_dir = exp_dir / "weights"
    last_pt = weights_dir / "last.pt"
    best_pt = weights_dir / "best.pt"
    train_done_flag = exp_dir / "TRAIN_COMPLETE.flag"
    eval_done_flag = exp_dir / "EVAL_COMPLETE.flag"

    start_time = time.time()

    try:
        # Train mới hoặc resume nếu phiên trước bị gián đoạn
        if not train_done_flag.exists():
            if last_pt.exists():
                print("Phát hiện last.pt -> resume training.")
                model = YOLO(str(last_pt))
                train_results = model.train(resume=True, workers=4)
            else:
                print(f"Train mới: {weights}, imgsz={imgsz}")
                model = YOLO(weights)

                train_results = model.train(
                    **TRAIN_ARGS,
                    imgsz=imgsz,
                    project=str(RUNS_ROOT),
                    name=exp_name,
                    exist_ok=True,
                )

            train_done_flag.touch()
            print("Training hoàn tất.")
        else:
            print("Training đã hoàn tất trước đó -> bỏ qua.")

        # Kiểm tra best.pt
        if not best_pt.exists():
            raise FileNotFoundError(f"Không tìm thấy best.pt: {best_pt}")

        # Load best model để evaluate
        best_model = YOLO(str(best_pt))

        print("Evaluate trên VALID...")
        val_metrics = best_model.val(
            data=str(LOCAL_YAML),
            split="val",
            imgsz=imgsz,
            batch=16,
            device=0,
            workers=4,
            plots=True,
            project=str(exp_dir),
            name="val_eval",
        )

        print("Evaluate trên TEST...")
        test_metrics = best_model.val(
            data=str(LOCAL_YAML),
            split="test",
            imgsz=imgsz,
            batch=16,
            device=0,
            workers=4,
            plots=True,
            project=str(exp_dir),
            name="test_eval",
        )

        # Lưu metric theo từng class
        val_per_class_csv = save_per_class_metrics(
            val_metrics, "val", exp_name, CLASS_NAMES
        )
        test_per_class_csv = save_per_class_metrics(
            test_metrics, "test", exp_name, CLASS_NAMES
        )

        # Copy best.pt ra folder models để dễ quản lý
        copied_best = MODELS_ROOT / f"{exp_name}_best.pt"
        shutil.copy2(best_pt, copied_best)

        # Tính số parameter và kích thước model
        param_count = sum(p.numel() for p in best_model.model.parameters())
        model_size_mb = copied_best.stat().st_size / (1024 ** 2)

        elapsed_min = (time.time() - start_time) / 60

        val_summary = extract_box_metrics(val_metrics)
        test_summary = extract_box_metrics(test_metrics)

        row = {
            "experiment": exp_name,
            "base_weights": weights,
            "imgsz": imgsz,
            "epochs_max": TRAIN_ARGS["epochs"],
            "val_precision": val_summary["precision"],
            "val_recall": val_summary["recall"],
            "val_mAP50": val_summary["mAP50"],
            "val_mAP50_95": val_summary["mAP50_95"],
            "test_precision": test_summary["precision"],
            "test_recall": test_summary["recall"],
            "test_mAP50": test_summary["mAP50"],
            "test_mAP50_95": test_summary["mAP50_95"],
            "params": param_count,
            "model_size_mb": round(model_size_mb, 3),
            "elapsed_min_this_run": round(elapsed_min, 2),
            "best_pt": str(copied_best),
            "val_per_class_csv": str(val_per_class_csv),
            "test_per_class_csv": str(test_per_class_csv),
        }

        summary_df = upsert_summary(row)
        eval_done_flag.touch()

        print("\nKẾT QUẢ:")
        print(pd.DataFrame([row]).T)

    except Exception as exc:
        errors[exp_name] = {
            "type": type(exc).__name__,
            "message": str(exc),
        }

        with open(ERROR_LOG, "w", encoding="utf-8") as f:
            json.dump(errors, f, ensure_ascii=False, indent=2)

        print(f"ERROR ở {exp_name}: {type(exc).__name__}: {exc}")
        print("Notebook sẽ chuyển sang experiment kế tiếp.")

    finally:
        # Dọn RAM/VRAM trước experiment kế tiếp
        try:
            del model
        except Exception:
            pass

        try:
            del best_model
        except Exception:
            pass

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

print("\n" + "=" * 90)
print("ĐÃ KẾT THÚC VÒNG OVERNIGHT TRAINING")
print("=" * 90)
print("Summary:", SUMMARY_CSV)
print("Error log:", ERROR_LOG if ERROR_LOG.exists() else "Không có lỗi được ghi.")



[1/4] EXPERIMENT: yolo11n_640
Training đã hoàn tất trước đó -> bỏ qua.
Evaluate trên VALID...
Ultralytics 8.4.142  Python-3.13.2 torch-2.14.0+cu126 CUDA:0 (NVIDIA GeForce RTX 4060, 8187MiB)
YOLO11n summary (fused): 100 layers, 2,593,462 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 2014.4487.2 MB/s, size: 176.1 KB)
val: Scanning D:\Project\TrafficSignAI\VR-TSD-2\valid\labels.cache... 1259 images, 1 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1259/1259 330.0Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 79/79 10.5it/s 7.5s0.2s
                   all       1259       1916      0.868      0.781      0.852      0.706
102-cam-di-nguoc-chieu        181        187      0.952      0.848       0.91       0.71
          103a-cam-oto         17         17      0.764          1       0.99      0.807
  103b-cam-oto-re-phai          2          2      0.923        0.5      0.503      0.303
   

## 9. Xếp hạng model sau khi train

Cell này đọc `overnight_summary.csv` và xếp hạng theo `test_mAP50_95`.

Khi chọn model cho Android, không nên chỉ lấy model có mAP cao nhất. Ngày mai ta sẽ so thêm:

- mAP / Precision / Recall
- kích thước `.pt`
- input 320 hay 640
- khả năng export LiteRT
- latency trên Galaxy A05s

Do đó bảng này là **kết quả phía AI**, chưa phải quyết định cuối cùng cho mobile.


In [9]:
# Hiển thị bảng xếp hạng cuối cùng
import pandas as pd

if SUMMARY_CSV.exists():
    summary = pd.read_csv(SUMMARY_CSV)

    summary = summary.sort_values(
        by="test_mAP50_95",
        ascending=False
    ).reset_index(drop=True)

    display_columns = [
        "experiment",
        "imgsz",
        "test_precision",
        "test_recall",
        "test_mAP50",
        "test_mAP50_95",
        "model_size_mb",
        "params",
    ]

    display(summary[display_columns])
else:
    print("Chưa có overnight_summary.csv")


,experiment,imgsz,test_precision,test_recall,test_mAP50,test_mAP50_95,model_size_mb,params
0,yolo11s_640,640,0.950797,0.924849,0.969630,0.804516,18.337,9450238
1,yolo11n_640,640,0.922132,0.843105,0.929526,0.756673,5.245,2601150
2,yolo11s_320,320,0.917825,0.851263,0.901483,0.709645,18.301,9450238
3,yolo11n_320,320,0.766237,0.709237,0.741643,0.577623,5.207,2601150


## 10. Tìm class yếu nhất của model tốt nhất

Do dataset có class imbalance khá rõ, cần xem những class nào có mAP thấp dù metric tổng thể tốt.

Cell này lấy model có `test_mAP50_95` cao nhất rồi hiển thị 15 class yếu nhất trên test set.


In [10]:
# Xem các class yếu nhất của model đứng đầu
if SUMMARY_CSV.exists():
    summary = pd.read_csv(SUMMARY_CSV).sort_values(
        by="test_mAP50_95",
        ascending=False
    )

    if len(summary) > 0:
        best_row = summary.iloc[0]
        per_class_path = Path(best_row["test_per_class_csv"])

        print("Best experiment:", best_row["experiment"])

        if per_class_path.exists():
            per_class = pd.read_csv(per_class_path)

            weak_classes = per_class.sort_values(
                by="mAP50_95",
                ascending=True
            ).head(15)

            display(weak_classes)
        else:
            print("Không tìm thấy:", per_class_path)


Best experiment: yolo11s_640


,class_id,class_name,precision,recall,mAP50,mAP50_95
4,4,106-cam-oto-tai,1.000000,0.772253,0.889682,0.576731
27,27,134-het-han-che-toc-do-toi-da,0.972738,0.956522,0.955000,0.598000
55,55,437-bat-dau-duong-cao-toc,NaN,NaN,NaN,0.618570
54,54,423a-duong-nguoi-di-bo-sang-ngang-1,NaN,NaN,NaN,0.631255
7,7,123b-cam-re-phai,0.960709,0.877551,0.927110,0.709070
39,39,225-chu-y-tre-em,0.883278,1.000000,0.995000,0.725327
16,16,127-toc-do-toi-da-50,0.934036,0.944021,0.966229,0.729538
48,48,302a-huong-phai-di-vong-chuong-ngai-vat-1,0.979203,0.535057,0.866646,0.732408
38,38,224-duong-nguoi-di-bo-cat-ngang,0.928712,0.965517,0.967769,0.738371
42,42,301a-cac-xe-chi-duoc-di-thang,0.988219,0.843039,0.941458,0.745500


## 11. Kiểm tra nhanh prediction của model tốt nhất

Cell này lấy ngẫu nhiên vài ảnh test và lưu prediction để ngày mai xem model có nhận diện trực quan hợp lý hay không.

Ultralytics sẽ lưu ảnh kết quả vào folder `runs/overnight_compare/final_predictions/`.


In [11]:
# Predict vài ảnh test bằng model tốt nhất
import random

if SUMMARY_CSV.exists():
    summary = pd.read_csv(SUMMARY_CSV).sort_values(
        by="test_mAP50_95",
        ascending=False
    )

    if len(summary) > 0:
        best_row = summary.iloc[0]
        best_model_path = best_row["best_pt"]

        test_image_dir = DATASET_ROOT / "test" / "images"
        test_images = [
            p for p in test_image_dir.iterdir()
            if p.suffix.lower() in IMAGE_EXTENSIONS
        ]

        sample_images = random.sample(
            test_images,
            k=min(12, len(test_images))
        )

        best_model = YOLO(best_model_path)

        best_model.predict(
            source=[str(p) for p in sample_images],
            imgsz=int(best_row["imgsz"]),
            conf=0.25,
            device=0,
            save=True,
            project=str(RUNS_ROOT),
            name="final_predictions",
            exist_ok=True,
        )

        print("Đã lưu prediction tại:", RUNS_ROOT / "final_predictions")



0: 640x640 1 303-noi-giao-nhau-chay-theo-vong-xuyen, 5.4ms
1: 640x640 1 303-noi-giao-nhau-chay-theo-vong-xuyen, 5.4ms
2: 640x640 1 302a-huong-phai-di-vong-chuong-ngai-vat-1, 5.4ms
3: 640x640 (no detections), 5.4ms
4: 640x640 1 127-toc-do-toi-da-50, 2 127-toc-do-toi-da-60s, 1 130-cam-dung-va-do-xe, 5.4ms
5: 640x640 1 102-cam-di-nguoc-chieu, 1 302a-huong-phai-di-vong-chuong-ngai-vat-1, 5.4ms
6: 640x640 1 106-cam-oto-tai, 1 123a-cam-re-trai, 5.4ms
7: 640x640 1 127-toc-do-toi-da-70, 5.4ms
8: 640x640 1 124b-cam-oto-quay-dau-xe, 1 225-chu-y-tre-em, 5.4ms
9: 640x640 1 102-cam-di-nguoc-chieu, 1 302a-huong-phai-di-vong-chuong-ngai-vat-1, 5.4ms
10: 640x640 1 124b-cam-oto-quay-dau-xe, 1 221-duong-go-ghe, 1 423a-duong-nguoi-di-bo-sang-ngang-1, 5.4ms
11: 640x640 1 420-bat-dau-khu-dong-dan-cu, 5.4ms
Speed: 1.6ms preprocess, 5.4ms inference, 1.0ms postprocess per image at shape (1, 3, 640, 640)
Results saved to D:\Project\TrafficSignAI\runs\overnight_compare\final_predictions
Đã lưu prediction tại: 

## 12. Snapshot môi trường sau khi pipeline chạy ổn

Cell này tạo `requirements-lock.txt` bằng `pip freeze`. Nó giúp tái tạo gần chính xác environment hiện tại về sau.

Không nên đưa folder `.venv` lên Git; giữ lại source code và file requirements là đủ.


In [12]:
# Freeze toàn bộ package/version hiện tại
import subprocess
import sys

freeze_output = subprocess.check_output(
    [sys.executable, "-m", "pip", "freeze"],
    text=True,
)

lock_path = PROJECT_ROOT / "requirements-lock.txt"
lock_path.write_text(freeze_output, encoding="utf-8")

print("Đã tạo:", lock_path)


Đã tạo: D:\Project\TrafficSignAI\requirements-lock.txt
